In [6]:
import pandas as pd
from sklearn.metrics import root_mean_squared_error
import pickle as pkl

In [36]:
# load in the test data
with open('../data/processed/df_tabular_test.pkl', 'rb') as f:

    df_test = pkl.load(f)

In [37]:
# drop columns that are not lag 1 hour or 1 week

cols_to_drop = df_test.columns.difference(set(('lag_1hr', 'lag_1wk', 'hourly_usage_kwh')))
df_test.drop(columns=cols_to_drop, inplace=True)

In [38]:
# create new feature which is the average usage at that hour and day of week for all time

# get datetimes
datetimes = pd.to_datetime(df_test.index.get_level_values('datetime'))

df_test['hour_day_avg'] = df_test.groupby([pd.Grouper(level='client_id'),
                                           datetimes.hour,
                                           datetimes.dayofweek
                                           ])['hourly_usage_kwh'].transform('mean')

In [39]:
df_test.head()

hourly_usage_kwh    lag_1hr    lag_1wk  \
client_id datetime                                                      
15        2014-03-14 20:00:00         62.370868  51.782025  68.827479   
          2014-03-14 21:00:00         62.758264  62.370868  63.662190   
          2014-03-14 22:00:00         55.914256  62.758264  56.172521   
          2014-03-14 23:00:00         51.394628  55.914256  50.103306   
          2014-03-15 00:00:00         43.646694  51.394628  45.971074   

                               hour_day_avg  
client_id datetime                           
15        2014-03-14 20:00:00     56.458456  
          2014-03-14 21:00:00     57.045701  
          2014-03-14 22:00:00     56.021866  
          2014-03-14 23:00:00     53.925005  
          2014-03-15 00:00:00     46.047939

In [40]:
# compute the rmse for each client
client_rmse_1hr_lag = df_test.groupby(level='client_id').apply(lambda g: root_mean_squared_error(g['hourly_usage_kwh'], g['lag_1hr']))
client_rmse_1wk_lag = df_test.groupby(level='client_id').apply(lambda g: root_mean_squared_error(g['hourly_usage_kwh'], g['lag_1wk']))
client_rmse_hour_day_avg = df_test.groupby(level='client_id').apply(lambda g: root_mean_squared_error(g['hourly_usage_kwh'], g['hour_day_avg']))

In [42]:
# compute mean usages across test data
mean_usages = df_test.groupby(level='client_id')['hourly_usage_kwh'].mean()

In [43]:
client_nrmse_1hr_lag = client_rmse_1hr_lag/mean_usages
client_nrmse_1wk_lag = client_rmse_1wk_lag/mean_usages
client_nrmse_hour_day_avg = client_rmse_hour_day_avg/mean_usages

In [44]:
# compute summary stats

summary_dict_1hr_lag = {
                'mean': client_nrmse_1hr_lag.mean(),
                'std': client_nrmse_1hr_lag.std(),
                'max': client_nrmse_1hr_lag.max(),
                'min': client_nrmse_1hr_lag.min(),
                }

summary_dict_1wk_lag = {
                'mean': client_nrmse_1wk_lag.mean(),
                'std': client_nrmse_1wk_lag.std(),
                'max': client_nrmse_1wk_lag.max(),
                'min': client_nrmse_1wk_lag.min(),
                }

summary_dict_hour_day_avg = {
                'mean': client_nrmse_hour_day_avg.mean(),
                'std': client_nrmse_hour_day_avg.std(),
                'max': client_nrmse_hour_day_avg.max(),
                'min': client_nrmse_hour_day_avg.min(),
                }

In [47]:
import json

with open('../logs/naive/nrmse_summary_stats_1hr_lag.json', 'w') as f:
    json.dump(summary_dict_1hr_lag, f, indent=4)

with open('../logs/naive/nrmse_summary_stats_1wk_lag.json', 'w') as f:
    json.dump(summary_dict_1wk_lag, f, indent=4)

with open('../logs/naive/nrmse_summary_stats_hour_day_avg.json', 'w') as f:
    json.dump(summary_dict_hour_day_avg, f, indent=4)


with open('../logs/naive/nrmse_per_client_1hr_lag.json', 'w') as f:
    json.dump(client_nrmse_1hr_lag.to_list(), f, indent=4)

with open('../logs/naive/nrmse_per_client_1wk_lag.json', 'w') as f:
    json.dump(client_nrmse_1wk_lag.to_list(), f, indent=4)

with open('../logs/naive/nrmse_per_client_hour_day_avg.json', 'w') as f:
    json.dump(client_nrmse_hour_day_avg.to_list(), f, indent=4)